# Wan2.1 (1.3B): First Spark, 2 sahne
1) **approach:** karanlık ormanda kamera uzaktan, ciddi, hafif ilk insana benzeyen şempanzeye yaklaşıyor
2) **strike:** taşları vuruyor, gerçekçi çakmaktaşı kıvılcımları çıkıyor, gözleri şaşkınlıkla açılıyor (aydınlık sahne)

**Colab:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU** → Kaydet → Çalışma zamanı → **Tümünü çalıştır**.
**Kaggle:** Settings → Accelerator **GPU T4**, Internet **On** → **Run All**.

Bitince `first_spark_clips.zip` iner. Klip başına yaklaşık 10–20 dk.

In [ ]:
!pip -q install -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg
import torch; print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
import torch, time, os
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
MODEL = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()   # T4'ün 16 GB belleğine sığdırır
print('model hazır')

In [ ]:
import os, time, torch
assert 'pipe' in globals(), 'Önce üstteki model hücresini çalıştır (en kolayı: Çalışma zamanı → Tümünü çalıştır)'
SCENES = {
    "approach": (
        "Cinematic slow dolly-in from far away toward a lone serious chimpanzee with a slightly early-human face, "
        "crouching on rocky ground in a dark prehistoric forest at dusk, holding a grey stone in each hand, calm intense focused stare, "
        "realistic fur detail, light fog, moody low light, faint cool moonlight rim light, documentary film, 35mm"
    ),
    "strike": (
        "Close-up of a serious chimpanzee with a slightly early-human face striking two flint stones together, "
        "real flint sparks fly out: a few tiny white-hot particles leaving short bright orange streaks, falling fast and fading within a second, physically accurate, like a real flint and steel strike filmed at high speed, the chimpanzee's eyes widen in surprise, "
        "warm golden sunlight from behind, soft realistic lighting, bright clear image, shaded rock wall background so the sparks glow, "
        "slow motion, realistic fur detail, sharp focus, documentary film"
    ),
}
NEG = ("cartoon, anime, 3d render, illustration, text, watermark, logo, subtitles, blurry, low quality, jpeg artifacts, "
       "deformed, extra limbs, extra fingers, static frame, worst quality, human clothing")
NEG_SCENE = {"approach": "", "strike": ", dark, underexposed, murky, gloomy, fireworks, magic sparkles, glitter, cartoon sparks, lightning, explosion, CGI particles"}

W, H, FRAMES = 480, 832, 49   # dikey, 49 kare yaklaşık 3 sn (16 fps)
SEEDS = [42, 7]               # her sahneden 2 farklı deneme, en iyisini seçeriz
os.makedirs('clips', exist_ok=True)
TOTAL = len(SCENES) * len(SEEDS); done = 0
for name, prompt in SCENES.items():
    for seed in SEEDS:
        print(f'▶ {name} (seed {seed}) üretiliyor…  Genel ilerleme: {done}/{TOTAL} klip – %{round(100*done/TOTAL)}')
        t0 = time.time()
        out = pipe(prompt=prompt, negative_prompt=NEG + NEG_SCENE[name], width=W, height=H, num_frames=FRAMES,
                   guidance_scale=5.0, num_inference_steps=30,
                   generator=torch.Generator('cpu').manual_seed(seed)).frames[0]
        export_to_video(out, f'clips/{name}_seed{seed}.mp4', fps=16)
        done += 1
        print(f'✅ {name} (seed {seed}) bitti: {round((time.time()-t0)/60, 1)} dk  |  Genel ilerleme: {done}/{TOTAL} klip – %{round(100*done/TOTAL)}')

Daha fazla deneme istersen `SEEDS` listesine başka sayılar ekle (örn. `[42, 7, 123]`).

In [ ]:
import shutil; shutil.make_archive('first_spark_clips', 'zip', 'clips')
try:
    from google.colab import files; files.download('first_spark_clips.zip')
except ImportError:
    print('Kaggle: sağdaki Output bölümünden first_spark_clips.zip dosyasını indir')